# Notebook 04 — Biểu đồ & Lưu kết quả
**Nhóm 67 | Tuần 2**

Vẽ 2 biểu đồ bắt buộc và lưu toàn bộ kết quả lên Drive.

> Chạy notebook 02 và 03 trước.

In [ ]:
import json, ast, os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

BASE    = '/content/nhom67'
OUT_DIR = f'{BASE}/results'

with open(f'{OUT_DIR}/student_simulation.json', encoding='utf-8') as f:
    sim = json.load(f)
rows = sim['submissions']

print(f'✓ Đọc {len(rows)} submissions')

## Biểu đồ 1 — Test Pass Rate: Public vs Hidden (RQ1)

In [ ]:
ids  = [r['sv_id']   for r in rows]
pub  = [r['pub_tpr'] for r in rows]
hid  = [r['hid_tpr'] for r in rows]

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(ids))

ax.bar([i - 0.2 for i in x], pub, 0.35,
       label='Bộ public (3 test/bài)', color='#378ADD', alpha=.85)
ax.bar([i + 0.2 for i in x], hid, 0.35,
       label='Bộ hidden (6 test/bài)', color='#1D9E75', alpha=.85)

for i, r in enumerate(rows):
    if r['is_false_positive']:
        ax.annotate('FP', xy=(i + 0.2, hid[i] + 1.5),
                    ha='center', fontsize=9,
                    color='#E24B4A', fontweight='bold')

ax.set_xticks(list(x))
ax.set_xticklabels(ids, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Test Pass Rate (%)')
ax.set_title(
    'Biểu đồ 1: So sánh Test Pass Rate — Bộ public vs Bộ hidden (RQ1)\n'
    'Bài đánh dấu FP = pass 3/3 public nhưng fail ít nhất 1 hidden test',
    fontsize=11
)
ax.legend(fontsize=10)
ax.set_ylim(0, 118)
ax.axhline(100, ls='--', color='gray', lw=0.8, alpha=0.5)
ax.grid(axis='y', alpha=0.3)

fp_total  = sim['fp_count']
fpr_total = sim['fpr_rate_%']
ax.text(0.98, 0.97,
        f'FPR = {fpr_total}% ({fp_total}/{len(rows)} bài)',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, color='#E24B4A',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#E24B4A', alpha=0.8))

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/bieu_do_1_tpr.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Lưu: results/bieu_do_1_tpr.png')

## Biểu đồ 2 — Phân loại lỗi: Public vs Hidden (RQ3)

In [ ]:
from collections import Counter

def count_errors(col):
    c = Counter()
    for r in rows:
        try:    d = ast.literal_eval(r[col])
        except: d = {}
        for k, v in d.items():
            c[k] += v
    return c

pub_err = count_errors('pub_errors')
hid_err = count_errors('hid_errors')

labels  = ['SE', 'WA', 'RE', 'TLE']
xlabels = ['Syntax Error', 'Wrong Answer', 'Runtime Error', 'Time Limit']
pub_v   = [pub_err.get(k, 0) for k in labels]
hid_v   = [hid_err.get(k, 0) for k in labels]

fig, ax = plt.subplots(figsize=(8, 5))
xb = range(len(labels))

b1 = ax.bar([i - 0.2 for i in xb], pub_v, 0.35,
            label='Bộ public', color='#378ADD', alpha=.85)
b2 = ax.bar([i + 0.2 for i in xb], hid_v, 0.35,
            label='Bộ hidden', color='#1D9E75', alpha=.85)

for bar in b1:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
for bar in b2:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

ax.set_xticks(list(xb))
ax.set_xticklabels(xlabels, fontsize=10)
ax.set_ylabel('Số lượng lỗi')
ax.set_title(
    'Biểu đồ 2: Phân loại lỗi — Bộ public vs Bộ hidden (RQ3)\n'
    'Bộ hidden phát hiện nhiều lỗi WA và RE hơn bộ public',
    fontsize=11
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/bieu_do_2_loi.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Lưu: results/bieu_do_2_loi.png')

## Lưu toàn bộ kết quả lên Google Drive

In [ ]:
import shutil

DRIVE_OUT = '/content/drive/MyDrive/nhom67_tuan2/results'
os.makedirs(DRIVE_OUT, exist_ok=True)

files = [
    'bieu_do_1_tpr.png',
    'bieu_do_2_loi.png',
    'baseline_summary.csv',
    'student_simulation.csv',
    'student_simulation.json',
    'metric_summary.json',
]

for fname in files:
    src = f'{OUT_DIR}/{fname}'
    dst = f'{DRIVE_OUT}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  ✓ {fname}')
    else:
        print(f'  ✗ Chưa có: {fname} — chạy notebook 02 và 03 trước')

print(f'\n✅ Đã lưu lên Drive: {DRIVE_OUT}')